# Lab 07: Chain Composition -- Solution

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

llm = ChatOllama(model="llama3.2:1b")

## Step 1: Two Independent Chains

Chain 1 generates a topic. Chain 2 writes about it.

In [ ]:
topic_prompt = ChatPromptTemplate.from_template(
    "Give me one specific, interesting topic about {subject}. Reply with just the topic name, nothing else."
)
topic_chain = topic_prompt | llm | StrOutputParser()

topic = topic_chain.invoke({"subject": "space exploration"})
print(f"Generated topic: {topic}")

In [ ]:
write_prompt = ChatPromptTemplate.from_template(
    "Write a 3-sentence explanation of: {topic}"
)
write_chain = write_prompt | llm | StrOutputParser()

explanation = write_chain.invoke({"topic": topic})
print(f"Explanation: {explanation}")

## Step 2: Connect Chains with Lambda

In [ ]:
full_chain = (
    topic_chain
    | (lambda topic: {"topic": topic})
    | write_chain
)
result = full_chain.invoke({"subject": "artificial intelligence"})
print(f"Result: {result}")

## Step 3: Three-Stage Pipeline

In [ ]:
simplify_prompt = ChatPromptTemplate.from_template(
    "Rewrite this so a 10-year-old can understand it. Use simple words:\n\n{text}"
)
simplify_chain = simplify_prompt | llm | StrOutputParser()

three_stage = (
    topic_chain
    | (lambda topic: {"topic": topic})
    | write_chain
    | (lambda explanation: {"text": explanation})
    | simplify_chain
)
result = three_stage.invoke({"subject": "quantum computing"})
print(f"Kid-friendly: {result}")

## Step 4: RunnablePassthrough

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Be concise. One sentence."),
    ("human", "{question}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()

qa_chain = RunnableParallel(
    question=RunnablePassthrough(),
    answer=answer_chain,
)
result = qa_chain.invoke({"question": "What is Docker?"})
print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")

## TODO 1: Topic -> Quiz Pipeline

In [ ]:
quiz_prompt = ChatPromptTemplate.from_template(
    "Create a multiple-choice quiz question about: {topic}\n"
    "Format: Question, then A), B), C), D) options."
)
quiz_chain = quiz_prompt | llm | StrOutputParser()

topic_to_quiz = (
    topic_chain
    | (lambda t: {"topic": t})
    | quiz_chain
)
print(topic_to_quiz.invoke({"subject": "Python programming"}))

## TODO 2: Translate -> Verify Pipeline

In [ ]:
to_hindi = ChatPromptTemplate.from_template(
    "Translate to Hindi. Return only the translation:\n{text}"
)
to_english = ChatPromptTemplate.from_template(
    "Translate to English. Return only the translation:\n{text}"
)

hindi_chain = to_hindi | llm | StrOutputParser()
english_chain = to_english | llm | StrOutputParser()

original = "Technology makes life easier"
hindi = hindi_chain.invoke({"text": original})
back = english_chain.invoke({"text": hindi})

print(f"Original:         {original}")
print(f"Hindi:            {hindi}")
print(f"Back to English:  {back}")

## TODO 3: RunnableParallel -- Parallel Analysis

In [ ]:
text_to_analyze = "Python is an amazing programming language. It makes development fast and fun. I love using it for data science and machine learning projects."

sentiment_prompt = ChatPromptTemplate.from_template(
    "Reply with exactly one word - Positive, Negative, or Neutral:\n{text}"
)
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize in one sentence:\n{text}"
)
topics_prompt = ChatPromptTemplate.from_template(
    "List 3 key topics as comma-separated words:\n{text}"
)

parallel_analysis = RunnableParallel(
    sentiment=sentiment_prompt | llm | StrOutputParser(),
    summary=summary_prompt | llm | StrOutputParser(),
    topics=topics_prompt | llm | StrOutputParser(),
)

result = parallel_analysis.invoke({"text": text_to_analyze})
print(f"Sentiment: {result['sentiment']}")
print(f"Summary: {result['summary']}")
print(f"Topics: {result['topics']}")

## Key Takeaways

- Lambda bridges transform data between chains
- Multi-stage pipelines: `chain1 | transform | chain2 | ...`
- `RunnablePassthrough` passes data through unchanged
- `RunnableParallel` runs multiple chains simultaneously
- Complex agents are built from simple, composable chains